In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install statsmodels networkx plotly -q
# import paper3_funder_network

In [ ]:
#!/usr/bin/env python3
"""
paper3_funder_network.py  —  Google Colab
=========================================
Wire the funding-enrichment output (fund_pairs) into Paper 3's two funding
deliverables:

  (A) FUNDER-NATIONALITY -> PRODUCER-NATIONALITY FLOW
      Bipartite, fractionally weighted: funder country -> author country.
      Exports an edge list (CSV), an adjacency matrix, a Plotly Sankey (reads
      as "who funds whom"), and a GEXF for Gephi chord layout.

  (B) THE INFERENTIAL TEST
      Does EXTERNAL funder nationality predict ABSENCE of local senior (last)
      authorship? Per-paper logistic model, GEE with study-country clusters
      (population-average, reviewer-standard), run separately per bloc.
      Key nuance (Rees 2021): capacity-building funders (e.g. US-gov Fogarty)
      behaved OPPOSITE to mere US co-authorship — so we separate "local funder
      present" from "external funder present" rather than lumping "has funding".

INPUT: the table that has BOTH fund_pairs AND the authorship/study-location
columns (i.e. with_funding.parquet).

SETUP
  !pip install pandas numpy statsmodels networkx plotly -q
"""

import re, sys
import numpy as np
import pandas as pd

# IN_PARQUET = "/content/drive/MyDrive/.../with_funding.parquet"
IN_PARQUET = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/funding_enrich.parquet"
OUT_DIR    = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/paper3_funding"

MAGHREB = {"MA","DZ","TN","LY","EG","MR"}
SSA = {"AO","BJ","BW","BF","BI","CV","CM","CF","TD","KM","CG","CD","CI","GQ","ER",
       "SZ","ET","GA","GM","GH","GN","GW","KE","LS","LR","MG","MW","ML","MZ","NA",
       "NE","NG","RW","ST","SN","SC","SL","SO","ZA","SS","SD","TZ","TG","UG","ZM","ZW"}


def S(x):
    return "" if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA else str(x)

def to_set(s, sep=";"):
    return {x for x in S(s).split(sep) if x and x.lower() != "nan"}

def parse_pairs(s):
    """'NIH::US; Wellcome Trust::GB' -> [('NIH','US'), ('Wellcome Trust','GB')]."""
    out = []
    for chunk in S(s).split(";"):
        chunk = chunk.strip()
        if "::" in chunk:
            name, ctry = chunk.split("::", 1)
            out.append((name.strip(), ctry.strip()))
    return out


# =========================================================================== #
# (A) FUNDER-COUNTRY -> PRODUCER-COUNTRY FLOW                                  #
# =========================================================================== #
def build_flow(df, producer_col="all_author_countries"):
    """Fractionally weighted edges funder_country -> author_country."""
    from collections import defaultdict
    edges = defaultdict(float)
    for _, row in df.iterrows():
        fcs = {c for _, c in parse_pairs(row.get("fund_pairs")) if c}
        pcs = to_set(S(row.get(producer_col)).replace("|", ";"))
        if not fcs or not pcs:
            continue
        w = 1.0 / (len(fcs) * len(pcs))     # don't let big teams/consortia dominate
        for fc in fcs:
            for pc in pcs:
                edges[(fc, pc)] += w
    edf = pd.DataFrame([{"funder_country": a, "producer_country": b, "weight": round(w, 4)}
                        for (a, b), w in edges.items()])
    edf = edf.sort_values("weight", ascending=False).reset_index(drop=True)
    edf.to_csv(f"{OUT_DIR}/funder_producer_edges.csv", index=False)
    print(f"[flow] {len(edf):,} edges -> funder_producer_edges.csv", file=sys.stderr)
    return edf

def flow_matrix(edf):
    m = edf.pivot_table(index="funder_country", columns="producer_country",
                        values="weight", aggfunc="sum", fill_value=0)
    m.to_csv(f"{OUT_DIR}/funder_producer_matrix.csv")
    return m

def sankey(edf, top=15, path=None):
    import plotly.graph_objects as go
    top_f = edf.groupby("funder_country")["weight"].sum().nlargest(top).index
    top_p = edf.groupby("producer_country")["weight"].sum().nlargest(top).index
    e = edf.copy()
    e["funder_country"] = np.where(e["funder_country"].isin(top_f), e["funder_country"], "Other")
    e["producer_country"] = np.where(e["producer_country"].isin(top_p), e["producer_country"], "Other")
    e = e.groupby(["funder_country", "producer_country"], as_index=False)["weight"].sum()
    fnodes = [f"F:{c}" for c in sorted(e['funder_country'].unique())]
    pnodes = [f"P:{c}" for c in sorted(e['producer_country'].unique())]
    nodes = fnodes + pnodes
    idx = {n: i for i, n in enumerate(nodes)}
    fig = go.Figure(go.Sankey(
        node=dict(label=nodes, pad=12, thickness=14),
        link=dict(source=[idx[f"F:{r.funder_country}"] for r in e.itertuples()],
                  target=[idx[f"P:{r.producer_country}"] for r in e.itertuples()],
                  value=[r.weight for r in e.itertuples()])))
    fig.update_layout(title="Funder nationality (F) -> Producer nationality (P)",
                      font_size=11)
    if path:
        fig.write_html(path); print(f"[sankey] {path}", file=sys.stderr)
    return fig

def export_gexf(edf, path):
    import networkx as nx
    G = nx.DiGraph()
    for r in edf.itertuples():
        G.add_node(f"F:{r.funder_country}", kind="funder")
        G.add_node(f"P:{r.producer_country}", kind="producer")
        G.add_edge(f"F:{r.funder_country}", f"P:{r.producer_country}", weight=r.weight)
    nx.write_gexf(G, path)
    print(f"[gexf] {path}  (open in Gephi for a chord/circular layout)", file=sys.stderr)


# =========================================================================== #
# (B) INFERENTIAL TEST — external funding vs local senior authorship          #
# =========================================================================== #
def bloc_of(iso):
    return "maghreb" if iso in MAGHREB else ("ssa" if iso in SSA else "other")

def team_size(row):
    for c in ("au_n",):
        v = row.get(c)
        if pd.notna(v) and S(v).isdigit():
            return int(v)
    pos = S(row.get("au_position_vec"))
    if pos:
        return len(pos.split("|"))
    return len(to_set(S(row.get("all_author_countries")).replace("|", ";"))) or 1

def build_reg_frame(df):
    recs = []
    for _, row in df.iterrows():
        target = to_set(row.get("study_countries"))
        if not target:
            continue
        allc = to_set(S(row.get("all_author_countries")).replace("|", ";"))
        if not allc:
            continue  # no resolved author country -> can't define local authorship
        last = to_set(row.get("last_author_countries"))
        fcs = {c for _, c in parse_pairs(row.get("fund_pairs")) if c}

        local_funder = bool(fcs & target)
        external_funder = bool(fcs - target)
        if not fcs:
            fund_cat = "none"
        elif local_funder and not external_funder:
            fund_cat = "local_only"
        elif external_funder and not local_funder:
            fund_cat = "external_only"
        else:
            fund_cat = "mixed"

        blocs = to_set(row.get("study_bloc"))
        recs.append({
            "oa_id": row["oa_id"],
            "local_last": int(bool(last & target)),
            "fund_cat": fund_cat,
            "external_funder": int(external_funder),
            "local_funder": int(local_funder),
            "log_authors": np.log(team_size(row)),
            "year_c": pd.to_numeric(row.get("oa_year"), errors="coerce"),
            "study_country": sorted(target)[0],        # cluster id
            "about_ssa": "ssa" in blocs,
            "about_maghreb": "maghreb" in blocs,
        })
    f = pd.DataFrame(recs).dropna(subset=["year_c"])
    f["year_c"] = f["year_c"] - f["year_c"].mean()
    return f

def gee_test(frame, cohort_mask, label):
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
    sub = frame[cohort_mask].copy()
    if sub["study_country"].nunique() < 3 or len(sub) < 50:
        print(f"\n[{label}] too few papers/clusters (n={len(sub)})"); return None
    sub["fund_cat"] = pd.Categorical(sub["fund_cat"],
        categories=["local_only", "none", "external_only", "mixed"])  # ref=local_only
    try:
        m = smf.gee("local_last ~ C(fund_cat) + log_authors + year_c",
                    groups="study_country", data=sub,
                    family=sm.families.Binomial(),
                    cov_struct=sm.cov_struct.Independence()).fit()
    except Exception as e:
        print(f"\n[{label}] model failed: {e}"); return None
    res = pd.DataFrame({"OR": np.exp(m.params),
                        "CI_low": np.exp(m.conf_int()[0]),
                        "CI_high": np.exp(m.conf_int()[1]),
                        "p": m.pvalues.round(4)})
    print(f"\n=== {label}  (n_papers={len(sub):,}, "
          f"n_countries={sub['study_country'].nunique()}) ===")
    print(f"reference = local_only funding; OR<1 means LESS local senior authorship")
    print(res.round(3).to_string())
    return res

def descriptive(frame, cohort_mask, label):
    sub = frame[cohort_mask]
    t = sub.groupby("fund_cat").agg(n=("local_last", "size"),
                                    local_last_share=("local_last", "mean"))
    t["local_last_share"] = t["local_last_share"].round(3)
    print(f"\n[{label}] local senior-authorship share by funding type:")
    print(t.to_string())


# =========================================================================== #
if __name__ == "__main__":
    import os; os.makedirs(OUT_DIR, exist_ok=True)
    df = pd.read_parquet(IN_PARQUET)
    print(f"[in] {len(df):,} rows", file=sys.stderr)

    # (A) flow — built on the bloc-relevant subset (about SSA or Maghreb)
    edf = build_flow(df)
    flow_matrix(edf)
    sankey(edf, path=f"{OUT_DIR}/funder_producer_sankey.html")
    export_gexf(edf, f"{OUT_DIR}/funder_producer.gexf")

    # (B) inferential test, per bloc
    frame = build_reg_frame(df)
    for label, mask in [("about SSA", frame["about_ssa"]),
                        ("about Maghreb", frame["about_maghreb"])]:
        descriptive(frame, mask, label)
        gee_test(frame, mask, label)